# Three configurable tasks

Follow [the experiment guide](../../../docs/leadership-evidence-lab/EXPERIMENTS.md). Fictional empathy classification, GoEmotions emotion classification, and EmpatheticDialogues response generation are different tasks, not one empathy benchmark. Downloads, model execution, training, and test evaluation are opt-in. Keep outputs cleared before publishing.

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
workshop = next((base for root in candidates for base in (root, root / 'workshops' / 'leadership-evidence-lab') if (base / 'empathy_lab').is_dir()), None)
if workshop is None:
    raise RuntimeError('Open this notebook from the repository or workshop directory')
sys.path.insert(0, str(workshop))
from empathy_lab.corpora import load_experiment, prepare
from empathy_lab.experiment import baseline, evaluate, train, validate


In [ ]:
PROFILE = 'goemotions-local'  # or fictional-local / empatheticdialogues-local
RUN = PROFILE + '-notebook-01'  # change for a fresh attempt; outputs are not overwritten
ALLOW_DOWNLOADS = False
RUN_MODELS = False
RUN_TRAINING = False
RUN_TEST = False
config = load_experiment(workshop / 'configs' / 'experiments' / (PROFILE + '.json'))
{'task': config.task, 'model': config.model.model_id, 'limits': config.limits}


## Prepare and validate

For public data, first review the source license, then explicitly allow downloads or acquire the pinned source through the CLI. This cell writes only ignored local prepared data and reports counts/fingerprints, not conversations.

In [ ]:
manifest = prepare(config, download=ALLOW_DOWNLOADS)
validate(config, tokens=RUN_MODELS)


In [ ]:
baseline(config, 'validation', run=RUN)


In [ ]:
if RUN_MODELS:
    base_validation = evaluate(config, 'validation', run=RUN)
    print(base_validation['metrics'])
if RUN_TRAINING:
    training = train(config, run=RUN)
    print({'steps': training['steps'], 'seconds': training['training_seconds']})
    adapted_validation = evaluate(config, 'validation', adapted=True, run=RUN)
    print(adapted_validation['metrics'])


## Freeze choices before the final test

Do not enable test evaluation to tune a model or repair prompts. The small default subsets verify engineering only. Generation token overlap is not empathy quality; emotion labels are not empathy-mechanism scores. Repeated turns from one dialogue are correlated.

In [ ]:
if RUN_TEST:
    print(baseline(config, 'test', run=RUN)['metrics'])
    if RUN_MODELS:
        print(evaluate(config, 'test', run=RUN)['metrics'])
    if RUN_TRAINING:
        print(evaluate(config, 'test', adapted=True, run=RUN)['metrics'])
